In [78]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import unicodedata


In [79]:
df1=pd.read_csv("../data/terrasses-autorisations.csv", sep=";")
df2=pd.read_csv("../data/dans-ma-rue.csv", sep=";")

In [80]:
df2['adresse']=df2['adresse'].str.split(',').str[0]

In [81]:
df2['adresse']

0                    16 rue Victor Duruy
1             31 Boulevard Saint-Jacques
2                          2 Rue d'Orsel
3                    63 Quai de la Seine
4          8 Boulevard de Bonne Nouvelle
                       ...              
1474280                     21 Rue Titon
1474281            16 Avenue Léon Bollée
1474282           236 Boulevard Voltaire
1474283                    2 Rue Emeriau
1474284            199 Rue du Chevaleret
Name: adresse, Length: 1474285, dtype: object

In [82]:
df2['adresse']= df2['adresse'].str.upper()
df1['adresse']= df1['adresse'].str.upper()

In [83]:
df2['adresse']=df2['adresse'].str.strip()
df1['adresse']=df1['adresse'].str.strip()

# Retirer les accents

In [84]:
def strip_accents(s):
    if pd.isna(s):
        return s
    nfkd = unicodedata.normalize('NFKD', s)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

df1['adresse'] = df1['adresse'].apply(strip_accents)
df2['adresse'] = df2['adresse'].apply(strip_accents)

# Collapse des espaces multiples en un seul espace

In [85]:
df1['adresse'] = df1['adresse'].str.replace(r'\s+', ' ', regex=True)
df2['adresse'] = df2['adresse'].str.replace(r'\s+', ' ', regex=True)

# Re-passer upper + strip apres ces transformations, sur les deux (idempotent, juste pour securiser)

In [86]:
# Verifier les virgules multiples dans l'adresse ORIGINALE de dans-ma-rue 
# -> necessite de relire la colonne brute puisque df2['adresse'] a deja ete splitee

In [87]:
tmp = pd.read_csv("../data/dans-ma-rue.csv", sep=";", usecols=["adresse"])
print(tmp['adresse'].str.count(',').value_counts())

adresse
1    1474251
2         31
0          3
Name: count, dtype: int64


# Mesure du recouvrement apres nettoyage

In [88]:
adresses_terrasses = set(df1['adresse'].dropna().unique())
adresses_dmr = set(df2['adresse'].dropna().unique())
communes = adresses_terrasses & adresses_dmr

print(f"Adresses uniques terrasses    : {len(adresses_terrasses)}")
print(f"Adresses uniques dans-ma-rue  : {len(adresses_dmr)}")
print(f"Adresses communes             : {len(communes)}")
print(f"Taux de recouvrement (cote terrasses) : {len(communes)/len(adresses_terrasses):.1%}")

Adresses uniques terrasses    : 14862
Adresses uniques dans-ma-rue  : 152698
Adresses communes             : 12021
Taux de recouvrement (cote terrasses) : 80.9%


In [89]:
df1['adresse'].duplicated().sum()

np.int64(9344)

In [90]:
df2['adresse'].duplicated().sum()

np.int64(1321587)

In [91]:
df1['periode_installation'].unique()

<StringArray>
[                nan,     'Toute l'année', 'du 11/04 Au 10/10',
 'du 01/10 Au 31/03', 'du 01/04 Au 30/09', 'du 15/03 Au 15/09',
 'du 31/12 Au 30/12', 'du 10/03 Au 10/09', 'du 31/05 Au 30/11']
Length: 9, dtype: str

In [92]:
df1.groupby('adresse').size()

adresse
- 143 BOULEVARD LEFEBVRE    1
0 RUE DU MONT CENIS         1
01 RUE DOMREMY              1
1 ALLEE DARIUS MILHAUD      1
1 AVENUE D'ITALIE           2
                           ..
RUE DU PELICAN              1
RUE FERDINAND DUVAL         1
RUE SAINT DOMINIQUE         1
RUE SURCOUF                 1
SSSS                        1
Length: 14862, dtype: int64

In [93]:
tailles = df1.groupby('adresse').size()
# print((tailles > 1).sum())        # nombre d'adresses avec 2+ terrasses
# print(tailles.value_counts())     # repartition complete : combien d'adresses ont 1, 2, 3... 
print(tailles)

adresse
- 143 BOULEVARD LEFEBVRE    1
0 RUE DU MONT CENIS         1
01 RUE DOMREMY              1
1 ALLEE DARIUS MILHAUD      1
1 AVENUE D'ITALIE           2
                           ..
RUE DU PELICAN              1
RUE FERDINAND DUVAL         1
RUE SAINT DOMINIQUE         1
RUE SURCOUF                 1
SSSS                        1
Length: 14862, dtype: int64


In [94]:
# 1. Adresses avec une seule terrasse (attribution non ambigue)
tailles = df1.groupby('adresse').size()
adresses_uniques = tailles[tailles == 1].index

# 2. On filtre df1 sur ces adresses
df1_clean = df1[df1['adresse'].isin(adresses_uniques)]

# 3. Inner join avec df2
df_join = df1_clean.merge(df2, on='adresse', how='inner', suffixes=('_terrasse', '_signalement'))

print(df_join.shape)

(94785, 28)


In [95]:
df_join

,typologie,adresse,arrondissement_terrasse,nom_enseigne,nom_societe,siret,longueur,largeur,periode_installation,lien_affichette,...,conseilquartier,datedecl,anneedecl,moisdecl,prefixe,intervenant,id_dmr,geo_shape_signalement,geo_point_2d_signalement,mois_annee_decla
0,TERRASSE FERMÉE,79 AVENUE DE SEGUR,75015.0,FÉLIX RESTAURANT,NaN,8.193063e+13,12.12,2.0,NaN,https://eudonet-terrasses.apps.paris.fr/xrm/at...,...,CAMBRONNE - GARIBALDI,2025-01-17,2025,1,IOS application,DPE-STPP-DT,A2025A087574,"{""coordinates"": [2.305659300528242, 48.8473969...","48.847396998797045, 2.305659300528242",2025-01
1,TERRASSE FERMÉE,79 AVENUE DE SEGUR,75015.0,FÉLIX RESTAURANT,NaN,8.193063e+13,12.12,2.0,NaN,https://eudonet-terrasses.apps.paris.fr/xrm/at...,...,CAMBRONNE - GARIBALDI,2025-12-07,2025,12,Back Office,DPE-STPP-DT,B2025L034905,"{""coordinates"": [2.3059764429120357, 48.847391...","48.84739154061818, 2.3059764429120357",2025-12
2,TERRASSE FERMÉE,79 AVENUE DE SEGUR,75015.0,FÉLIX RESTAURANT,NaN,8.193063e+13,12.12,2.0,NaN,https://eudonet-terrasses.apps.paris.fr/xrm/at...,...,CAMBRONNE - GARIBALDI,2025-07-29,2025,7,IOS application,DPE-STPP-DT,A2025G158025,"{""coordinates"": [2.305866500111666, 48.8475569...","48.84755699895653, 2.305866500111666",2025-07
3,TERRASSE FERMÉE,79 AVENUE DE SEGUR,75015.0,FÉLIX RESTAURANT,NaN,8.193063e+13,12.12,2.0,NaN,https://eudonet-terrasses.apps.paris.fr/xrm/at...,...,CAMBRONNE - GARIBALDI,2026-04-09,2026,4,Android,DPMP-UEV,G2026D046423,"{""coordinates"": [2.3058523998415947, 48.847346...","48.84734699861689, 2.3058523998415947",2026-04
4,TERRASSE FERMÉE,79 AVENUE DE SEGUR,75015.0,FÉLIX RESTAURANT,NaN,8.193063e+13,12.12,2.0,NaN,https://eudonet-terrasses.apps.paris.fr/xrm/at...,...,CAMBRONNE - GARIBALDI,2025-01-17,2025,1,IOS application,DPE-STPP-DT,A2025A083234,"{""coordinates"": [2.3058953002415192, 48.847594...","48.84759499864925, 2.3058953002415192",2025-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94780,CONTRE TERRASSE ESTIVALE SUR STATIONNEMENT,46 AVENUE EMILE ZOLA,75015.0,LES DÉLICES D'ORIENT,NaN,4.021462e+13,10.00,1.7,NaN,NaN,...,EMERIAU - ZOLA,2025-11-24,2025,11,IOS application,DVD,A2025K118769,"{""coordinates"": [2.282710000207423, 48.8462029...","48.8462029989377, 2.282710000207423",2025-11
94781,CONTRE TERRASSE ESTIVALE SUR STATIONNEMENT,36 RUE D'AUTEUIL,75016.0,GINETTE,NaN,9.286980e+13,6.00,1.7,NaN,NaN,...,AUTEUIL NORD,2026-04-03,2026,4,Page DMR Paris.fr,DPMP-Divisions,S2026D014327,"{""coordinates"": [2.2660340000566834, 48.847693...","48.8476939991117, 2.2660340000566834",2026-04
94782,CONTRE TERRASSE ESTIVALE SUR STATIONNEMENT,36 RUE D'AUTEUIL,75016.0,GINETTE,NaN,9.286980e+13,6.00,1.7,NaN,NaN,...,AUTEUIL NORD,2026-03-16,2026,3,Page DMR Paris.fr,DPMP-Divisions,S2026C090245,"{""coordinates"": [2.2660340000566834, 48.847693...","48.8476939991117, 2.2660340000566834",2026-03
94783,TERRASSE ESTIVALE SUR TROTTOIR FACE À LA DEVAN...,117 AVENUE DE LA BOURDONNAIS,75007.0,HÔTEL LE CERCLE,NaN,4.340117e+13,6.00,2.6,NaN,NaN,...,GROS CAILLOU,2025-01-11,2025,1,Android,DPE-STPP-DT,G2025A051493,"{""coordinates"": [2.3047825996387297, 48.854773...","48.854773999264296, 2.3047825996387297",2025-01


In [73]:
# Tableau croisé : effectifs bruts
contingence = pd.crosstab(df_join['typologie'], df_join['type'])
print(contingence)

# Meme tableau mais en % par ligne, plus lisible pour comparer les typologies entre elles
print(pd.crosstab(df_join['typologie'], df_join['type'], normalize='index'))

# Test statistique
chi2, p, dof, expected = chi2_contingency(contingence)
print(f"p-value : {p}")

type                                                Activités commerciales et professionnelles  \
typologie                                                                                        
COMMERCE ACCESSOIRE                                                                          1   
CONTRE ETALAGE                                                                               0   
CONTRE TERRASSE ESTIVALE SUR PLACES ET TERRE-PLEIN                                          19   
CONTRE TERRASSE ESTIVALE SUR STATIONNEMENT                                                1256   
CONTRE TERRASSE ESTIVALE SUR TROTTOIR DÉSAXÉE P...                                           3   
CONTRE TERRASSE ESTIVALE SUR TROTTOIR FACE À LA...                                         188   
CONTRE TERRASSE SUR TROTTOIR                                                                22   
CONTRE TERRASSE SUR VOIE PIÉTONNE                                                           15   
CONTRE ÉTALAGE SUR P

In [74]:
# n = nombre total de signalements dans le tableau croise
n = contingence.sum().sum()

# r = nombre de typologies de terrasse differentes (lignes du tableau)
# c = nombre de types de signalement differents (colonnes du tableau)
r, c = contingence.shape

# formule du V de Cramer
cramers_v = (chi2 / (n * min(r - 1, c - 1))) ** 0.5

print(f"n (total signalements) : {n}")
print(f"r (typologies)         : {r}")
print(f"c (types signalement)  : {c}")
print(f"V de Cramer            : {cramers_v:.3f}")

n (total signalements) : 94774
r (typologies)         : 29
c (types signalement)  : 10
V de Cramer            : 0.075


In [75]:
adresses_terrasses = set(df1['adresse'].dropna().unique())
df2['a_une_terrasse'] = df2['adresse'].isin(adresses_terrasses)

contingence_h1 = pd.crosstab(df2['a_une_terrasse'], df2['type'])
print(contingence_h1)

chi2_h1, p_h1, dof_h1, expected_h1 = chi2_contingency(contingence_h1)

n_h1 = contingence_h1.sum().sum()
r_h1, c_h1 = contingence_h1.shape
cramers_v_h1 = (chi2_h1 / (n_h1 * min(r_h1 - 1, c_h1 - 1))) ** 0.5

print(f"p-value H1     : {p_h1}")
print(f"V de Cramer H1 : {cramers_v_h1:.3f}")

type            Activités commerciales et professionnelles  \
a_une_terrasse                                               
False                                                17596   
True                                                 13316   

type            Arbres, végétaux et animaux  \
a_une_terrasse                                
False                                 16335   
True                                   2322   

type            Autos, motos, vélos, trottinettes...  Dégradation du sol  \
a_une_terrasse                                                             
False                                          76322                   1   
True                                           10180                   0   

type             Eau  Graffitis, tags, affiches et autocollants  \
a_une_terrasse                                                    
False           7048                                     297183   
True             948                                    

In [96]:

print("NaN dans typologie :", df1['typologie'].isna().sum())
print("NaN dans type (df2) :", df2['type'].isna().sum())
print("NaN dans adresse df1 :", df1['adresse'].isna().sum())
print("NaN dans adresse df2 :", df2['adresse'].isna().sum())


NaN dans typologie : 10
NaN dans type (df2) : 0
NaN dans adresse df1 : 8
NaN dans adresse df2 : 0


In [97]:
print("Lignes dupliquees df1 :", df1.duplicated().sum())
print("Lignes dupliquees df2 :", df2.duplicated().sum())

Lignes dupliquees df1 : 0
Lignes dupliquees df2 : 0


In [98]:
incoherent = df_join['arrondissement_terrasse'] != df_join['arrondissement_signalement']
print(f"Lignes incoherentes : {incoherent.sum()} sur {len(df_join)} ({incoherent.mean():.2%})")

Lignes incoherentes : 94785 sur 94785 (100.00%)


In [99]:
print(df1['arrondissement'].dtype, df2['arrondissement'].dtype)
print(df_join[['arrondissement_terrasse', 'arrondissement_signalement']].dtypes)
print(df_join[['arrondissement_terrasse', 'arrondissement_signalement']].head(10))


float64 int64
arrondissement_terrasse       float64
arrondissement_signalement      int64
dtype: object
   arrondissement_terrasse  arrondissement_signalement
0                  75015.0                          15
1                  75015.0                          15
2                  75015.0                          15
3                  75015.0                          15
4                  75015.0                          15
5                  75015.0                          15
6                  75015.0                          15
7                  75015.0                          15
8                  75015.0                          15
9                  75015.0                          15


In [100]:
df_join['arrondissement_terrasse_num'] = (df_join['arrondissement_terrasse'] % 100).astype('Int64')

incoherent2 = df_join['arrondissement_terrasse_num'] != df_join['arrondissement_signalement']
print(f"Lignes incoherentes (apres correction) : {incoherent2.sum()} sur {len(df_join)} ({incoherent2.mean():.2%})")

Lignes incoherentes (apres correction) : 3202 sur 94785 (3.51%)


# H1-bis : densite de terrasses par arrondissement vs signalements commerciaux

Hypothese : les arrondissements avec une densite de terrasses plus elevee
presentent-ils aussi une densite plus elevee de signalements "Activites
commerciales et professionnelles" ?

Note : association ecologique au niveau arrondissement (agregats), pas un
effet demontre au niveau d'une terrasse individuelle -- a garder en tete
dans l'interpretation finale.

Etape 1 : verifier l'unite de la colonne `surface` de quartier_paris et la
coherence des codes d'arrondissement (`c_ar`) avant de calculer quoi que
ce soit.

In [ ]:
import geopandas as gpd

quartiers = gpd.read_file("../data/quartier_paris.geojson")

# Verifier l'unite de surface et la coherence des codes arrondissement
print(quartiers[['c_ar', 'surface']].describe())
print(sorted(quartiers['c_ar'].unique()))